In [ ]:
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JasonCiemielewski/SVEF-Drug-Rescue/blob/main/notebooks/model_creation.ipynb)

## Set Up

In [1]:
import os
import sys
import pandas as pd
import numpy as np
from datetime import datetime

# Detect Environment
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/BIFX546/'
    DATA_DIR = os.path.join(PROJECT_ROOT, 'data/demo/')
    sys.path.append(PROJECT_ROOT)
    !pip install -q matplotlib-venn
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    DATA_DIR = os.path.join(PROJECT_ROOT, 'data/demo/')
    sys.path.append(PROJECT_ROOT)

print(f"Project Root added to path: {PROJECT_ROOT}")
print(f"Data Directory: {DATA_DIR}")

Project Root added to path: c:\Users\Ownwer\Documents\BIFX\BIFX546\Final_Project
Data Directory: c:\Users\Ownwer\Documents\BIFX\BIFX546\Final_Project\data/demo/


In [2]:
# Run this inside the script file, not just the console

print(f"Is PROJECT_ROOT still here? {PROJECT_ROOT}")
print(f"Currently defined variables: {list(locals().keys())}")

Is PROJECT_ROOT still here? c:\Users\Ownwer\Documents\BIFX\BIFX546\Final_Project
Currently defined variables: ['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', 'open', 'help', '_', '__', '___', '_i', '_ii', '_iii', '_i1', 'os', 'sys', 'pd', 'np', 'datetime', 'IN_COLAB', 'PROJECT_ROOT', 'DATA_DIR', '_i2']


In [3]:
INTERIM_DIR = os.path.join(PROJECT_ROOT, 'data', 'interim')
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
RAW_DIR = os.path.join(PROJECT_ROOT, 'data','raw')

## **Filtering nct_id**

Trials are filtered to identify nct_id that have the following characteristics.  After each data file is filtered, a set of nct_id is created to subset the next datafile prior to further filtering  
**studies.txt**  
*overall_status*  
- COMPLETED
- UNKNOWN
- TERMINATED
- WITHDRAWN
- SUSPENDED
- ENROLLING_BY_INVITATION
- ACTIVE_NOT_RECRUITING

*study_type*  
- INTERVENTIONAL

(create nct_id_filter_set)

**design_groups.txt** 
*group_type*  
- EXPERIMENTAL

(create nct_id_filter_set2)

**interventions.txt**  
*intervention_type*  
- DRUG

(create nct_id_filter_set3)

### **studies.txt**

In [4]:
## set studies_features equal to the features to keep for subsetting the studies.txt RAW data
studies_features = ['nct_id', 'overall_status', 'phase', 'study_type', 'enrollment', 'enrollment_type', 'number_of_arms', 'why_stopped', 'is_fda_regulated_drug', 'is_fda_regulated_device']

## set status_list equal to overal_status to keep 
status_list = [
    'COMPLETED', 'UNKNOWN', 'TERMINATED', 'WITHDRAWN',
    'SUSPENDED', 'ENROLLING_BY_INVITATION', 'ACTIVE_NOT_RECRUITING'
]

In [5]:
# load studies.txt using only studies_features
studies_df = pd.read_csv(
    os.path.join(RAW_DIR, 'studies.txt'),
    sep = '|',
    usecols = studies_features
)

In [6]:
# Apply filters to studies_df
mask = (studies_df['study_type'] == 'INTERVENTIONAL') & (studies_df['overall_status'].isin(status_list))
studies_filtered_df = studies_df[mask].copy()

In [7]:
# verify filtering
print(f"Total trials in subset: {len(studies_filtered_df)}")
print("Unique statuses found:", studies_filtered_df['overall_status'].unique())

Total trials in subset: 374913
Unique statuses found: <ArrowStringArray>
[                'UNKNOWN',   'ACTIVE_NOT_RECRUITING',
               'COMPLETED',               'WITHDRAWN',
              'TERMINATED', 'ENROLLING_BY_INVITATION',
               'SUSPENDED']
Length: 7, dtype: str


In [8]:
## Create a set of the nct_id in studies_filtered_df for use in other RAW datasets
nct_id_filter_set = set(studies_filtered_df['nct_id'])
print(f"Filter set created with {len(nct_id_filter_set):,} unique IDs")

Filter set created with 374,913 unique IDs


In [9]:
#Extract studies_filtered_df column names into a list
studies_filtered_headers = studies_filtered_df.columns.tolist()

#Verify the list
print(f"Captured {len(studies_filtered_headers)} headers.")
print(studies_filtered_headers)

Captured 10 headers.
['nct_id', 'study_type', 'overall_status', 'phase', 'enrollment', 'enrollment_type', 'number_of_arms', 'why_stopped', 'is_fda_regulated_drug', 'is_fda_regulated_device']


In [10]:
# Remove mask, studies_df, studies_filtered_df
import gc

items_to_del = [
    'mask', 'studies_df', 'studies_filtered_df'
]

for item in items_to_del:
    # In a notebook, variables are stored in the globals() dictionary
    if item in globals():
        del globals()[item]
        print(f"Success: {item} was found and cleared from memory.")
    else:
        print(f"Notice: {item} was not found. No action needed.")

# Trigger the garbage collector once at the end to free up the RAM
gc.collect()

Success: mask was found and cleared from memory.
Success: studies_df was found and cleared from memory.
Success: studies_filtered_df was found and cleared from memory.


1276

### **design_groups.txt**

In [11]:
# Create an empty list to store the pieces that match your IDs
design_chunks = []

# Pass the path directly into the loop
for chunk in pd.read_csv(os.path.join(RAW_DIR, 'design_groups.txt'), 
                        sep='|', 
                        chunksize=50000, 
                        low_memory=False):
    
    # Filter: Keep only rows where the nct_id is in your pre-defined set
    filtered_chunk = chunk[chunk['nct_id'].isin(nct_id_filter_set)]
    
    if not filtered_chunk.empty:
        design_chunks.append(filtered_chunk)

# Combine results
design_groups_filtered_df = pd.concat(design_chunks, axis=0).reset_index(drop=True)

In [12]:
design_groups_filtered_df['group_type'].isna().sum() # 0

np.int64(0)

In [13]:
# Apply filters to design_groups_filtered_df
mask = (design_groups_filtered_df['group_type'] == 'EXPERIMENTAL')
design_groups_filtered_df2 = design_groups_filtered_df[mask].copy()

In [14]:
# Verify that ONLY Experimental groups remain
print(design_groups_filtered_df2['group_type'].value_counts())

group_type
EXPERIMENTAL    430224
Name: count, dtype: int64


In [15]:
# 1. Create the refined set of NCT IDs
nct_id_filter_set2 = set(design_groups_filtered_df2['nct_id'])

# 2. Quick validation
print(f"Original subset count: {len(nct_id_filter_set):,}")
print(f"Refined 'Experimental' count: {len(nct_id_filter_set2):,}")
print(f"Trials filtered out: {len(nct_id_filter_set) - len(nct_id_filter_set2):,}")

Original subset count: 374,913
Refined 'Experimental' count: 286,303
Trials filtered out: 88,610


In [17]:
# Remove 'chunk', 'design_groups_filtered_df', 'design_groups_filtered_df2',
#        'design_chunks' 
import gc

items_to_del = [
    'chunk', 'design_groups_filtered_df', 'design_groups_filtered_df2',
    'design_chunks'
]

for item in items_to_del:
    # In a notebook, variables are stored in the globals() dictionary
    if item in globals():
        del globals()[item]
        print(f"Success: {item} was found and cleared from memory.")
    else:
        print(f"Notice: {item} was not found. No action needed.")

# Trigger the garbage collector once at the end to free up the RAM
gc.collect()

Notice: chunk was not found. No action needed.
Notice: design_groups_filtered_df was not found. No action needed.
Notice: design_groups_filtered_df2 was not found. No action needed.
Success: design_chunks was found and cleared from memory.


89

### **interventions.txt**

In [18]:
# The "Sieve" Approach
drug_interventions_chunks = []

# Intake 50,000 rows at a time
for chunk in pd.read_csv(os.path.join(RAW_DIR, 'interventions.txt'), 
                        sep='|', 
                        chunksize=50000, 
                        usecols=['id', 'nct_id', 'intervention_type']):
    
    # Apply BOTH filters (Type and ID) while the chunk is on the "desk"
    mask = (chunk['intervention_type'].str.upper() == 'DRUG') & \
           (chunk['nct_id'].isin(nct_id_filter_set2))
    
    filtered_chunk = chunk[mask]
    
    if not filtered_chunk.empty:
        drug_interventions_chunks.append(filtered_chunk)

# Assemble only the relevant data
filtered_interventions_df = pd.concat(drug_interventions_chunks, axis=0).reset_index(drop=True)

In [19]:
# 1. Create the refined set of NCT IDs
nct_id_filter_set3 = set(filtered_interventions_df['nct_id'])

# 2. Quick validation
print(f"Original subset count: {len(nct_id_filter_set2):,}")
print(f"Refined 'Experimental' count: {len(nct_id_filter_set3):,}")
print(f"Trials filtered out: {len(nct_id_filter_set2) - len(nct_id_filter_set3):,}")

Original subset count: 286,303
Refined 'Experimental' count: 128,186
Trials filtered out: 158,117


In [20]:
# Remove chunk, filtered_chunk, filtered_interventions_df,
#    mask, drug_interventions_chunks 
import gc

items_to_del = [
    'chunk', 'filtered_chunk', 'filtered_interventions_df',
    'mask', 'drug_interventions_chunks'
]

for item in items_to_del:
    # In a notebook, variables are stored in the globals() dictionary
    if item in globals():
        del globals()[item]
        print(f"Success: {item} was found and cleared from memory.")
    else:
        print(f"Notice: {item} was not found. No action needed.")

# Trigger the garbage collector once at the end to free up the RAM
gc.collect()

Success: chunk was found and cleared from memory.
Success: filtered_chunk was found and cleared from memory.
Success: filtered_interventions_df was found and cleared from memory.
Success: mask was found and cleared from memory.
Success: drug_interventions_chunks was found and cleared from memory.


204

## **Safety Features**

In [21]:
rep_event_features = [
    'id', 'nct_id', 'result_group_id', 'event_type', 'subjects_affected',
    'subjects_at_risk', 'description', 'event_count', 'adverse_event_term',
    'frequency_threshold', 'time_frame', 'ctgov_group_code'
]

In [22]:
# The "Sieve" Approach
rep_events_chunks = []

# Intake 50,000 rows at a time
for chunk in pd.read_csv(os.path.join(RAW_DIR, 'reported_events.txt'), 
                        sep='|', 
                        chunksize=50000, 
                        usecols=rep_event_features):
    
    # Apply BOTH filters (Type and ID) while the chunk is on the "desk"
    mask = (chunk['nct_id'].isin(nct_id_filter_set3))
    
    filtered_chunk = chunk[mask]
    
    if not filtered_chunk.empty:
        rep_events_chunks.append(filtered_chunk)

# Assemble only the relevant data
filtered_rep_events_df = pd.concat(rep_events_chunks, axis=0).reset_index(drop=True)

In [45]:
# Create a binary feature: Did the investigator actually report the count?
filtered_rep_events_df['is_intensity_reported'] = filtered_rep_events_df['event_count'].notna().astype(int)

In [46]:
filtered_rep_events_df.columns

Index(['id', 'nct_id', 'result_group_id', 'ctgov_group_code', 'time_frame',
       'event_type', 'subjects_affected', 'subjects_at_risk', 'description',
       'event_count', 'adverse_event_term', 'frequency_threshold',
       'is_intensity_reported'],
      dtype='str')

##### **Arm Level Aggregation**

In [59]:
#Filter for 'serious' events to focus on SAE (Serious Adverse Events)
# We isolate serious events because they are the primary safety signals for drug rescue
sae_only_df = filtered_rep_events_df[filtered_rep_events_df['event_type'] == 'serious'].copy()

In [72]:
# 2. Perform the Aggregation at the Arm Level (nct_id + result_group_id)
# Note: Use .max() for subjects_at_risk because it is the same for all events in an arm
arm_level_safety = sae_only_df.groupby(['nct_id', 'result_group_id', 'ctgov_group_code']).agg(
    total_sae_occurrences=('event_count', 'sum'),      # Total times SAEs happened (Intensity numerator)
    sum_subjects_affected=('subjects_affected', 'sum'), # Sum of people affected per term (Note: may have overlaps)
    arm_subjects_at_risk=('subjects_at_risk', 'max'),  # Total population of the arm (Incidence denominator)
    intensity_reported_count=('is_intensity_reported', 'sum'),
    total_sae_rows=('id', 'count')
).reset_index()

$$SAE \text{ Intensity} = \frac{\sum \text{Total SAE Occurrences}}{\sum \text{Subjects Affected}}$$

In [73]:
# 3. Calculate Safety Features for the Random Forest
# Calculate Intensity: Average number of SAEs per affected subject
arm_level_safety['sae_intensity'] = (
    arm_level_safety['total_sae_occurrences'] / arm_level_safety['sum_subjects_affected']
).replace([np.inf, -np.inf], np.nan).fillna(0)

$$SAE\text{ Incidence Rate} = \min\left(1.0, \frac{\sum \text{Subjects Affected}}{\text{Arm Subjects at Risk}}\right)$$

In [74]:
# Calculate Incidence: What percentage of the arm population experienced a serious event
# We use .clip(upper=1.0) because summing subjects_affected across different terms can exceed 100% due to overlaps
arm_level_safety['sae_incidence_rate'] = (
    arm_level_safety['sum_subjects_affected'] / arm_level_safety['arm_subjects_at_risk']
).clip(upper=1.0).fillna(0)

In [75]:
# Calculate Data Quality: Percentage of rows where intensity was actually reported
arm_level_safety['pct_intensity_reported'] = (
    arm_level_safety['intensity_reported_count'] / arm_level_safety['total_sae_rows']
).fillna(0)

In [76]:
print(f"Aggregated {len(sae_only_df):,} event rows into {len(arm_level_safety):,} arm-level profiles.")

Aggregated 3,292,517 event rows into 78,107 arm-level profiles.


In [ ]:
# Remove filtered_rep_events_df
import gc

# Check both global and local memory for the variable name
if 'chunk' in globals() or 'chunk' in locals():
    del chunk
    gc.collect() # Trigger the 'garbage truck' to pick up the freed RAM
    print("Success: chunk was found and cleared from memory.")
else:
    print("Notice: chunk was not found. No action needed.")

# Check both global and local memory for the variable name
if 'filtered_chunk' in globals() or 'filtered_chunk' in locals():
    del filtered_chunk
    gc.collect() # Trigger the 'garbage truck' to pick up the freed RAM
    print("Success: filtered_chunk was found and cleared from memory.")
else:
    print("Notice: filtered_chunk was not found. No action needed.")

# Check both global and local memory for the variable name
if 'rep_events_chunks' in globals() or 'rep_events_chunks' in locals():
    del rep_events_chunks
    gc.collect() # Trigger the 'garbage truck' to pick up the freed RAM
    print("Success: rep_events_chunks was found and cleared from memory.")
else:
    print("Notice: rep_events_chunks was not found. No action needed.")


# Check both global and local memory for the variable name
if 'mask' in globals() or 'mask' in locals():
    del mask
    gc.collect() # Trigger the 'garbage truck' to pick up the freed RAM
    print("Success: mask was found and cleared from memory.")
else:
    print("Notice: mask was not found. No action needed.")

Success: filtered_rep_events_df was found and cleared from memory.
Notice: chunk was not found. No action needed.
Notice: filtered_chunk was not found. No action needed.
Notice: rep_events_chunks was not found. No action needed.
Notice: mask was not found. No action needed.


##### **Mortality**

In [77]:
# Updated features for totals
totals_chunks = []
totals_features = ['nct_id', 'ctgov_group_code', 'event_type', 'subjects_affected', 'subjects_at_risk']

for chunk in pd.read_csv(os.path.join(RAW_DIR, 'reported_event_totals.txt'), 
                        sep='|', 
                        chunksize=50000, 
                        usecols=totals_features):
    
    # Using your existing filter set from the notebook
    mask = chunk['nct_id'].isin(nct_id_filter_set3)
    filtered_chunk = chunk[mask]
    
    if not filtered_chunk.empty:
        totals_chunks.append(filtered_chunk)

filtered_totals_df = pd.concat(totals_chunks, axis=0).reset_index(drop=True)

$$Mortality\ Rate_{arm} = \frac{\sum \text{Subjects\ Affected}_{(\text{event\_type} = \text{'deaths'})}}{\max(\text{Subjects\ at\ Risk}_{arm})}$$


In [78]:
# 1. Calculate mortality per arm using the correct 'deaths' label
# We use .strip() just in case there are hidden spaces in the strings
arm_mortality = filtered_totals_df[filtered_totals_df['event_type'].str.strip() == 'deaths'].groupby(['nct_id', 'ctgov_group_code']).agg(
    deaths=('subjects_affected', 'sum'),
    at_risk=('subjects_at_risk', 'max')
).reset_index()

# 2. Calculate the mortality rate feature
arm_mortality['mortality_rate'] = (arm_mortality['deaths'] / arm_mortality['at_risk']).fillna(0)

print(f"Success! Captured mortality data for {len(arm_mortality):,} experimental arms.")

Success! Captured mortality data for 110,684 experimental arms.


In [79]:
arm_mortality.describe()

,deaths,at_risk,mortality_rate
count,110684.000000,67571.000000,110684.000000
mean,4.332605,86.571651,0.073998
std,28.297907,1013.855677,0.208582
min,0.000000,0.000000,0.000000
25%,0.000000,7.000000,0.000000
50%,0.000000,21.000000,0.000000
75%,0.000000,60.000000,0.000000
max,1361.000000,242344.000000,1.000000


In [80]:
# 1. Check if the initial sieve actually caught anything
print(f"Total rows in filtered_totals_df: {len(filtered_totals_df):,}")

# 2. Check the exact labels used in the event_type column
if len(filtered_totals_df) > 0:
    print("Unique event types found:", filtered_totals_df['event_type'].unique())
else:
    print("WARNING: filtered_totals_df is empty. Check nct_id_filter_set3.")

# 3. Check a sample of nct_ids to see if they exist in the raw file
print("Sample of IDs we are looking for:", list(nct_id_filter_set3)[:5])

Total rows in filtered_totals_df: 332,052
Unique event types found: <ArrowStringArray>
['deaths', 'other', 'serious']
Length: 3, dtype: str
Sample of IDs we are looking for: ['NCT05133817', 'NCT01462227', 'NCT04536688', 'NCT01269047', 'NCT02506556']


In [81]:
# Join the Mortality data to your existing arm_level_safety table
# The join on both NCT_ID and CTGOV_GROUP_CODE prevents data collisions
final_arm_safety_profile = pd.merge(
    arm_level_safety, 
    arm_mortality[['nct_id', 'ctgov_group_code', 'deaths', 'mortality_rate']], 
    on=['nct_id', 'ctgov_group_code'], 
    how='left'
).fillna(0)

# Quick check of the final features
print(final_arm_safety_profile[['sae_incidence_rate', 'mortality_rate', 'sae_intensity']].head())

   sae_incidence_rate  mortality_rate  sae_intensity
0            0.522727             0.0       2.478261
1            0.526882             0.0       2.326531
2            0.709677             0.0       2.227273
3            0.560748             0.0       2.550000
4            0.574803             0.0       2.465753


### **Relational Bridge**

In [82]:
# The Sieve for the Bridge Table
bridge_chunks = []
# design_group_id matches the result_group_id from your events table
bridge_features = ['nct_id', 'design_group_id', 'intervention_id']

for chunk in pd.read_csv(os.path.join(RAW_DIR, 'design_group_interventions.txt'), 
                        sep='|', 
                        chunksize=50000, 
                        usecols=bridge_features):
    
    # Filter using your refined drug-trial set
    mask = chunk['nct_id'].isin(nct_id_filter_set3)
    filtered_chunk = chunk[mask]
    
    if not filtered_chunk.empty:
        bridge_chunks.append(filtered_chunk)

filtered_bridge_df = pd.concat(bridge_chunks, axis=0).reset_index(drop=True)

In [85]:
# Join the safety profile to the bridge
# This maps: Arm Metrics -> Intervention ID
drug_safety_mapped = pd.merge(
    final_arm_safety_profile,
    filtered_bridge_df,
    left_on=['nct_id', 'result_group_id'],
    right_on=['nct_id', 'design_group_id'],
    how='inner' # We only want arms that have a confirmed intervention
)

In [86]:
# 1. Check Data Types (The #1 cause of empty joins)
print("--- Data Type Check ---")
print(f"Safety ID Type (result_group_id): {final_arm_safety_profile['result_group_id'].dtype}")
print(f"Bridge ID Type (design_group_id): {filtered_bridge_df['design_group_id'].dtype}")

# 2. Check for Overlap in NCT IDs
safety_ncts = set(final_arm_safety_profile['nct_id'])
bridge_ncts = set(filtered_bridge_df['nct_id'])
overlap_ncts = safety_ncts.intersection(bridge_ncts)
print(f"\n--- ID Overlap Check ---")
print(f"NCT IDs in Safety: {len(safety_ncts):,}")
print(f"NCT IDs in Bridge: {len(bridge_ncts):,}")
print(f"NCT IDs appearing in BOTH: {len(overlap_ncts):,}")

# 3. Look at a Sample from both sides for one overlapping NCT ID
if len(overlap_ncts) > 0:
    sample_nct = list(overlap_ncts)[0]
    print(f"\n--- Sample for {sample_nct} ---")
    print("Safety Side IDs:", final_arm_safety_profile[final_arm_safety_profile['nct_id'] == sample_nct]['result_group_id'].tolist())
    print("Bridge Side IDs:", filtered_bridge_df[filtered_bridge_df['nct_id'] == sample_nct]['design_group_id'].tolist())

--- Data Type Check ---
Safety ID Type (result_group_id): int64
Bridge ID Type (design_group_id): int64

--- ID Overlap Check ---
NCT IDs in Safety: 27,003
NCT IDs in Bridge: 128,181
NCT IDs appearing in BOTH: 27,003

--- Sample for NCT00370331 ---
Safety Side IDs: [683356545, 683356546]
Bridge Side IDs: [356498399, 356498400]


In [88]:
# 1. Get Result Titles (Links your Safety Metrics to a name)
result_group_chunks = []
for chunk in pd.read_csv(os.path.join(RAW_DIR, 'result_groups.txt'), 
                        sep='|', 
                        chunksize=50000, 
                        usecols=['nct_id', 'id', 'title']):
    # Use your pre-defined drug-trial filter
    mask = chunk['nct_id'].isin(nct_id_filter_set3)
    filtered_chunk = chunk[mask]
    if not filtered_chunk.empty:
        result_group_chunks.append(filtered_chunk)

# Rename 'id' to 'result_group_id' to match your safety profile
result_titles_df = pd.concat(result_group_chunks).rename(columns={'id': 'result_group_id', 'title': 'arm_title'})

In [89]:
# 2. Get Design Titles (Links your drugs to a name)
design_group_chunks = []
for chunk in pd.read_csv(os.path.join(RAW_DIR, 'design_groups.txt'), 
                        sep='|', 
                        chunksize=50000, 
                        usecols=['nct_id', 'id', 'title']):
    mask = chunk['nct_id'].isin(nct_id_filter_set3)
    filtered_chunk = chunk[mask]
    if not filtered_chunk.empty:
        design_group_chunks.append(filtered_chunk)

# Rename 'id' to 'design_group_id' to match your bridge table
design_titles_df = pd.concat(design_group_chunks).rename(columns={'id': 'design_group_id', 'title': 'arm_title'})

In [90]:
# A. Build the "Rosetta Stone" by matching on NCT ID and Arm Title
universal_bridge = pd.merge(
    result_titles_df,
    design_titles_df,
    on=['nct_id', 'arm_title'],
    how='inner'
)

# B. Link your safety profile to this new bridge
safety_with_design_ids = pd.merge(
    final_arm_safety_profile,
    universal_bridge[['nct_id', 'result_group_id', 'design_group_id']],
    on=['nct_id', 'result_group_id'],
    how='inner'
)

# C. Finally, join to your Intervention IDs (The Drugs)
# Using filtered_bridge_df from your Cell 82
drug_safety_mapped = pd.merge(
    safety_with_design_ids,
    filtered_bridge_df[['nct_id', 'design_group_id', 'intervention_id']],
    on=['nct_id', 'design_group_id'],
    how='inner'
)

print(f"Success! Linked safety data for {drug_safety_mapped['intervention_id'].nunique():,} unique drugs.")

Success! Linked safety data for 28,557 unique drugs.


In [92]:
print(drug_safety_mapped.columns)

Index(['nct_id', 'result_group_id', 'ctgov_group_code',
       'total_sae_occurrences', 'sum_subjects_affected',
       'arm_subjects_at_risk', 'intensity_reported_count', 'total_sae_rows',
       'sae_intensity', 'sae_incidence_rate', 'pct_intensity_reported',
       'deaths', 'mortality_rate', 'design_group_id', 'intervention_id'],
      dtype='str')


In [93]:
len(drug_safety_mapped)

41960

In [94]:
# Save your progress!
drug_safety_mapped.to_csv(os.path.join(INTERIM_DIR, 'drug_safety_mapped.csv'), index=False)

In [96]:
# Remove arm_level_safety, arm_mortality, chunk, design_titles_df, 
##     filtered_bridge_df, filtered_chunk, filtered_rep_events_df, 
import gc

items_to_del = [
    'arm_level_safety', 'arm_mortality', 'chunk', 'design_titles_df',
    'filtered_bridge_df', 'filtered_chunk', 'filtered_rep_events_df',
    'filtered_totals_df', 'final_arm_safety_profile', 'mask', 'result_titles_df',
    'sae_only_df', 'safety_with_design_ids', 'universal_bridge'
]

for item in items_to_del:
    # In a notebook, variables are stored in the globals() dictionary
    if item in globals():
        del globals()[item]
        print(f"Success: {item} was found and cleared from memory.")
    else:
        print(f"Notice: {item} was not found. No action needed.")

# Trigger the garbage collector once at the end to free up the RAM
gc.collect()

Notice: arm_level_safety was not found. No action needed.
Notice: arm_mortality was not found. No action needed.
Notice: chunk was not found. No action needed.
Notice: design_titles_df was not found. No action needed.
Notice: filtered_bridge_df was not found. No action needed.
Notice: filtered_chunk was not found. No action needed.
Notice: filtered_rep_events_df was not found. No action needed.
Notice: filtered_totals_df was not found. No action needed.
Notice: final_arm_safety_profile was not found. No action needed.
Notice: mask was not found. No action needed.
Success: result_titles_df was found and cleared from memory.
Notice: sae_only_df was not found. No action needed.
Notice: safety_with_design_ids was not found. No action needed.
Notice: universal_bridge was not found. No action needed.


79

## **Efficacy Features**

In [97]:
# 1. Identify Primary Outcomes
primary_outcome_chunks = []
for chunk in pd.read_csv(os.path.join(RAW_DIR, 'outcomes.txt'), 
                        sep='|', 
                        chunksize=50000, 
                        usecols=['id', 'nct_id', 'outcome_type']):
    # Filter for your drug trials and 'primary' outcome types
    mask = (chunk['nct_id'].isin(nct_id_filter_set3)) & \
           (chunk['outcome_type'].str.lower() == 'primary')
    filtered_chunk = chunk[mask]
    if not filtered_chunk.empty:
        primary_outcome_chunks.append(filtered_chunk)

primary_outcomes_df = pd.concat(primary_outcome_chunks).rename(columns={'id': 'outcome_id'})

In [99]:
primary_outcomes_df.columns

Index(['outcome_id', 'nct_id', 'outcome_type'], dtype='str')

In [101]:
# Updated features to include non_inferiority_type
p_val_features = ['nct_id', 'outcome_id', 'non_inferiority_type', 'p_value']

p_val_chunks = []
for chunk in pd.read_csv(os.path.join(RAW_DIR, 'outcome_analyses.txt'), 
                        sep='|', 
                        chunksize=50000, 
                        usecols=p_val_features):
    mask = chunk['nct_id'].isin(nct_id_filter_set3)
    filtered_chunk = chunk[mask]
    if not filtered_chunk.empty:
        p_val_chunks.append(filtered_chunk)

raw_p_values_df = pd.concat(p_val_chunks)

In [103]:
# Join to Primary Outcomes (using your existing primary_outcomes_df)
# 1. Clean the p-values by forcing the column to string type first
# This prevents the AttributeError when encountering floats or NaNs
efficacy_bridge['p_value_clean'] = pd.to_numeric(
    efficacy_bridge['p_value'].astype(str).str.replace(r'[<>=\s]', '', regex=True), 
    errors='coerce'
)

# 2. Standardize the non_inferiority_type
# We also use .astype(str) here because NaNs in this column will cause the same error
efficacy_bridge['non_inferiority_type'] = (
    efficacy_bridge['non_inferiority_type']
    .astype(str)
    .str.title()
    .str.strip()
)

# Standardize the types (to handle case sensitivity)
efficacy_bridge['non_inferiority_type'] = efficacy_bridge['non_inferiority_type'].str.title().str.strip()

# DEFINE THE SIGNAL:
# A "Superiority Failure" is our prime rescue candidate (Stalled IP)
def label_failure(row):
    if pd.isna(row['p_value_clean']):
        return "Unknown"
    if row['non_inferiority_type'] == 'Superiority':
        return 1 if row['p_value_clean'] > 0.05 else 0
    elif row['non_inferiority_type'] == 'Non-Inferiority':
        # If a non-inferiority trial fails (p > 0.05), it's a Performance Failure
        return 2 if row['p_value_clean'] > 0.05 else 0
    return 0

efficacy_bridge['efficacy_failure_label'] = efficacy_bridge.apply(label_failure, axis=1)

In [104]:
# 1. Inspect the distribution of Efficacy Labels
print("--- Efficacy Label Distribution (Arm Level) ---")
print(efficacy_bridge['efficacy_failure_label'].value_counts())

# 2. Calculate percentages to see the "Rescue Opportunity"
print("\n--- Percentage Breakdown ---")
print(efficacy_bridge['efficacy_failure_label'].value_counts(normalize=True) * 100)

# 3. Optional: Map the labels back to a readable string for a quick check
label_map = {0: "Success (p <= 0.05)", 1: "Stalled IP (Superiority Failure)", 2: "Performance Failure (Inferiority)", "Unknown": "Data Missing"}
print("\n--- Human-Readable Summary ---")
print(efficacy_bridge['efficacy_failure_label'].map(label_map).value_counts())

--- Efficacy Label Distribution (Arm Level) ---
efficacy_failure_label
0          23590
Unknown    11985
1           5465
Name: count, dtype: int64

--- Percentage Breakdown ---
efficacy_failure_label
0          57.480507
Unknown    29.203216
1          13.316277
Name: proportion, dtype: float64

--- Human-Readable Summary ---
efficacy_failure_label
Success (p <= 0.05)                 23590
Data Missing                        11985
Stalled IP (Superiority Failure)     5465
Name: count, dtype: int64
